# 📖 Notebook 1: Understanding the Read Scaling Problem

Before optimizing, we need to understand why reads become a bottleneck and how to measure the problem.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why reads dominate most applications
- How to measure database performance
- What causes slow reads
- The read scaling progression

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [1]:
import psycopg2
import time
from concurrent.futures import ThreadPoolExecutor

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_connection()
    print("✅ Connected to PostgreSQL")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker compose up -d")

✅ Connected to PostgreSQL


## 📊 Read vs Write Ratios

Most applications have dramatically more reads than writes. Let's visualize this.

In [2]:
print("📊 Read/Write Ratios in Real Systems")
print("=" * 60)
print()

systems = [
    ("URL Shortener (Bitly)", 1000, 1, "1 URL created, 1000 clicks"),
    ("Social Media (Twitter)", 100, 1, "1 tweet, 100 views"),
    ("E-commerce (Amazon)", 500, 1, "1 product, 500 views"),
    ("Video Platform (YouTube)", 10000, 1, "1 upload, 10k views"),
    ("Banking App", 5, 1, "Check balance often, transact rarely"),
]

print(f"{'System':<25} {'Reads':>8} {'Writes':>8} {'Ratio':>10}   Notes")
print("-" * 80)

for system, reads, writes, notes in systems:
    ratio = f"{reads}:{writes}"
    bar = "█" * min(reads // 50, 20)
    print(f"{system:<25} {reads:>8} {writes:>8} {ratio:>10}   {notes}")

print()
print("💡 Key insight: Optimizing reads has 10-1000x more impact than writes!")

📊 Read/Write Ratios in Real Systems

System                       Reads   Writes      Ratio   Notes
--------------------------------------------------------------------------------
URL Shortener (Bitly)         1000        1     1000:1   1 URL created, 1000 clicks
Social Media (Twitter)         100        1      100:1   1 tweet, 100 views
E-commerce (Amazon)            500        1      500:1   1 product, 500 views
Video Platform (YouTube)     10000        1    10000:1   1 upload, 10k views
Banking App                      5        1        5:1   Check balance often, transact rarely

💡 Key insight: Optimizing reads has 10-1000x more impact than writes!


## 🐌 Demonstrating Slow Reads

Let's see how reads slow down without proper optimization.

In [3]:
def measure_query(query: str, description: str) -> float:
    conn = get_connection()
    cursor = conn.cursor()
    
    start = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = (time.time() - start) * 1000
    
    conn.close()
    return elapsed, len(results)

print("🐌 Query Performance Without Indexes")
print("=" * 60)
print()

queries = [
    ("SELECT * FROM users WHERE email = 'user50@example.com'", "Find user by email"),
    ("SELECT * FROM posts WHERE user_id IN (1, 2, 3, 4, 5)", "Find posts by top 5 users"),
    ("SELECT * FROM products WHERE category = 'Electronics'", "Find products by category"),
    (
        "SELECT u.username, p.content, p.like_count "
        "FROM posts p JOIN users u ON p.user_id = u.id "
        "WHERE u.follower_count > 5000 ORDER BY p.created_at DESC LIMIT 20",
        "Feed query (join + filter + sort)"
    ),
]

for query, description in queries:
    elapsed, count = measure_query(query, description)
    status = "⚡" if elapsed < 5 else "🐌" if elapsed < 50 else "💀"
    print(f"{status} {description}")
    print(f"   Time: {elapsed:.2f}ms | Rows: {count}")
    print()

🐌 Query Performance Without Indexes

⚡ Find user by email
   Time: 1.21ms | Rows: 1

⚡ Find posts by top 5 users
   Time: 1.19ms | Rows: 11

⚡ Find products by category
   Time: 4.68ms | Rows: 975

⚡ Feed query (join + filter + sort)
   Time: 2.60ms | Rows: 20



## 📈 Simulating Load

What happens when many users query simultaneously?

In [4]:
def run_query(query: str) -> float:
    conn = get_connection()
    cursor = conn.cursor()
    start = time.time()
    cursor.execute(query)
    cursor.fetchall()
    elapsed = time.time() - start
    conn.close()
    return elapsed * 1000

def load_test(query: str, concurrent_users: int, requests_per_user: int) -> dict:
    times = []
    
    def user_session():
        session_times = []
        for _ in range(requests_per_user):
            session_times.append(run_query(query))
        return session_times
    
    with ThreadPoolExecutor(max_workers=concurrent_users) as executor:
        futures = [executor.submit(user_session) for _ in range(concurrent_users)]
        for future in futures:
            times.extend(future.result())
    
    return {
        "total_requests": len(times),
        "avg_ms": sum(times) / len(times),
        "min_ms": min(times),
        "max_ms": max(times),
        "p95_ms": sorted(times)[int(len(times) * 0.95)]
    }

print("📈 Load Test: Find user by email")
print("=" * 60)
print()

query = "SELECT * FROM users WHERE email = 'user50@example.com'"

for users in [1, 5, 10, 20]:
    results = load_test(query, concurrent_users=users, requests_per_user=10)
    print(f"👥 {users} concurrent users:")
    print(f"   Avg: {results['avg_ms']:.2f}ms | P95: {results['p95_ms']:.2f}ms | Max: {results['max_ms']:.2f}ms")
    print()

📈 Load Test: Find user by email



👥 1 concurrent users:
   Avg: 1.35ms | P95: 1.79ms | Max: 1.79ms



👥 5 concurrent users:
   Avg: 0.84ms | P95: 1.30ms | Max: 1.42ms



👥 10 concurrent users:
   Avg: 0.98ms | P95: 1.89ms | Max: 2.57ms



👥 20 concurrent users:
   Avg: 1.23ms | P95: 2.46ms | Max: 4.08ms



## 🔍 Understanding EXPLAIN

PostgreSQL's `EXPLAIN` command shows how queries are executed.

In [5]:
def explain_query(query: str):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(f"EXPLAIN ANALYZE {query}")
    plan = cursor.fetchall()
    conn.close()
    return [row[0] for row in plan]

print("🔍 EXPLAIN: Finding user by email (no index)")
print("=" * 60)
print()

plan = explain_query("SELECT * FROM users WHERE email = 'user50@example.com'")
for line in plan:
    print(line)

print()
print("💡 Notice 'Seq Scan' - the database scans EVERY row!")

🔍 EXPLAIN: Finding user by email (no index)

Index Scan using idx_users_email_covering on users  (cost=0.29..8.30 rows=1 width=606) (actual time=0.009..0.010 rows=1 loops=1)
  Index Cond: ((email)::text = 'user50@example.com'::text)
Planning Time: 0.127 ms
Execution Time: 0.021 ms

💡 Notice 'Seq Scan' - the database scans EVERY row!


## 🗺️ The Read Scaling Progression

Here's the typical path for scaling reads:

In [6]:
print("🗺️ Read Scaling Progression")
print("=" * 60)
print()
print("""
Stage 1: OPTIMIZE DATABASE
┌─────────────────────────────────────────────────────────────┐
│  • Add indexes on frequently queried columns                │
│  • Optimize queries (avoid SELECT *, use LIMIT)            │
│  • Denormalize hot queries                                  │
│  • Use materialized views for aggregations                  │
│                                                             │
│  Handles: ~10,000 - 50,000 QPS                             │
└─────────────────────────────────────────────────────────────┘
                          │
                          ▼ Still not enough?
Stage 2: ADD CACHING
┌─────────────────────────────────────────────────────────────┐
│  • Application-level cache (Redis/Memcached)               │
│  • Cache frequently accessed data                           │
│  • Implement proper cache invalidation                      │
│                                                             │
│  Handles: ~100,000 - 500,000 QPS                           │
└─────────────────────────────────────────────────────────────┘
                          │
                          ▼ Still not enough?
Stage 3: SCALE HORIZONTALLY
┌─────────────────────────────────────────────────────────────┐
│  • Read replicas (leader-follower replication)             │
│  • Database sharding (for very large datasets)             │
│  • CDN for edge caching                                     │
│                                                             │
│  Handles: ~1,000,000+ QPS                                  │
└─────────────────────────────────────────────────────────────┘
""")

print("💡 Always start at Stage 1 - it's often enough!")

🗺️ Read Scaling Progression


Stage 1: OPTIMIZE DATABASE
┌─────────────────────────────────────────────────────────────┐
│  • Add indexes on frequently queried columns                │
│  • Optimize queries (avoid SELECT *, use LIMIT)            │
│  • Denormalize hot queries                                  │
│  • Use materialized views for aggregations                  │
│                                                             │
│  Handles: ~10,000 - 50,000 QPS                             │
└─────────────────────────────────────────────────────────────┘
                          │
                          ▼ Still not enough?
Stage 2: ADD CACHING
┌─────────────────────────────────────────────────────────────┐
│  • Application-level cache (Redis/Memcached)               │
│  • Cache frequently accessed data                           │
│  • Implement proper cache invalidation                      │
│                                                             │
│  Handles: ~100,00

## 🧪 Quick Quiz

1. **Why do reads typically outnumber writes?**

2. **What does 'Seq Scan' in EXPLAIN mean?**

3. **When should you jump straight to caching?**

In [7]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why reads outnumber writes:")
print("   - Content is created once, viewed many times")
print("   - Users browse more than they create")
print("   - Many users view same popular content")
print()
print("2. Seq Scan means:")
print("   - Sequential Scan - reads EVERY row in the table")
print("   - O(n) complexity - scales linearly with data")
print("   - Usually indicates a missing index")
print()
print("3. When to jump to caching:")
print("   - Almost NEVER skip database optimization!")
print("   - Caching adds complexity (invalidation, staleness)")
print("   - Only after indexes and query optimization")
print("   - Exception: Known hot data (celebrity profiles)")

📝 Quiz Answers

1. Why reads outnumber writes:
   - Content is created once, viewed many times
   - Users browse more than they create
   - Many users view same popular content

2. Seq Scan means:
   - Sequential Scan - reads EVERY row in the table
   - O(n) complexity - scales linearly with data
   - Usually indicates a missing index

3. When to jump to caching:
   - Almost NEVER skip database optimization!
   - Caching adds complexity (invalidation, staleness)
   - Only after indexes and query optimization
   - Exception: Known hot data (celebrity profiles)


## 📚 Summary

### Key Takeaways

1. **Reads dominate** - Most apps are 10:1 to 1000:1 read/write ratio
2. **Measure first** - Use EXPLAIN before optimizing
3. **Start simple** - Database optimization before adding infrastructure
4. **Seq Scan is bad** - Indicates missing index
5. **Load testing reveals truth** - Single query perf ≠ concurrent perf

### Next Up

In **Notebook 2**, we'll learn database optimization techniques:
- Creating effective indexes
- Query optimization
- Understanding index types